In [ ]:
%pip install catboost

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
# from catboost import CatBoostClassifier


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] < 1].sort_values('Missing_Percentage', ascending=False)
print(missing_data['Column'])
print("Missing Data Analysis:")
df = df[missing_data['Column']]
df.dropna(subset=missing_data['Column'])

In [ ]:
df.info()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# there are no categorical columns

In [ ]:
# Task 4: Write your code here:
target_column = "Target"

X = df.drop(target_column, axis=1)
y = df[target_column]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 5: Write your code here:
import seaborn as sns

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")
# it is imbalance

In [ ]:
# Task 1: Write your code here:
#already done
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)


In [ ]:
# Task 1: Write your code here:
print("\nTop 3 important features:")
importances = model.feature_importances_
for i in np.argsort(importances)[-3:][::-1]:
    print(f"  Feature {i}: {importances[i]:.3f}")

In [ ]:
sklearn_models = {


  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

In [ ]:
# Gather importances from the models (from the last fold)
importances = {}

importances['CatBoost'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1,2, figsize=(18, 20))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
print("\nTop 3 important features :")
importances = model.feature_importances_
for i in np.argsort(importances)[-1:][::-1]:
    print(f"  Feature D_45: {importances[i]:.3f}")

In [ ]:
# Task Bonus: Write your code here: